## RNN Encoder Decoder [Aakrit Pahadi: ACE080BCT002]

## Objectives

* To gain an understanding of the structure and operation of Recurrent Neural Networks (RNNs) within an Encoder-Decoder framework designed for sequence-to-sequence learning.
* To develop and implement an RNN-based Encoder-Decoder model capable of translating text from French to English.
* To study and apply the Bahdanau Attention mechanism, enabling the decoder to concentrate on the most relevant sections of the input sequence for improved translation accuracy.
* To learn the purpose and implementation of the Teacher Forcing technique during the training process of sequence-to-sequence neural networks.
* To assess the effectiveness of the trained model by testing its translation performance on previously unseen French sentences.


## Theory

### RNN Encoder-Decoder

The RNN Encoder-Decoder architecture is widely used for sequence-to-sequence applications, including machine translation. It is composed of two primary components:

**1. Encoder:**
The encoder is a Recurrent Neural Network (RNN), such as a Simple RNN, GRU, or LSTM, that processes the input sequence one word at a time. As it reads each word, it updates its hidden state, and the final hidden state serves as a context vector. This vector summarizes the important semantic information of the complete input sentence.

**2. Decoder:**
The decoder is another RNN that uses the encoder's context vector as its initial hidden state. It generates the translated output sequence one token at a time by predicting the next word while continuously updating its hidden state until the entire sentence is produced.

### Bahdanau Attention Mechanism

The Bahdanau Attention mechanism enhances the traditional Encoder-Decoder model by allowing the decoder to access all encoder hidden states during the decoding process instead of depending only on the final context vector. At every decoding step, an alignment score is calculated between the current decoder hidden state and each encoder hidden state. These scores are converted into attention weights, which are then used to compute a context vector as the weighted combination of all encoder hidden states. This enables the model to focus on the most relevant parts of the input sequence, resulting in more accurate translations, especially for longer sentences.

### Teacher Forcing

Teacher Forcing is a commonly used training technique for sequence-to-sequence models. During training, the decoder receives the actual target word from the previous time step as its next input rather than using its own predicted output. Providing the correct previous word helps the model learn more effectively, improves training stability, and accelerates convergence. Although this approach leads to better learning during training, it may introduce a difference between training and testing because, during inference, the decoder must rely entirely on its own predictions.


In [ ]:
# requirements
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [ ]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [ ]:
# Turn a Unicode string to plain ASCII, thanks to
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [ ]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [ ]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [ ]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [ ]:
PATH = r'/content/data.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words.

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['il lit', 'he is reading']


15

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

In [ ]:
# class DecoderRNN(nn.Module):
#     def __init__(self, hidden_size, output_size):
#         super(DecoderRNN, self).__init__()
#         self.embedding = nn.Embedding(output_size, hidden_size)
#         self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
#         self.out = nn.Linear(hidden_size, output_size)

#     def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
#         batch_size = encoder_outputs.size(0)
#         decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
#         decoder_hidden = encoder_hidden
#         decoder_outputs = []

#         for i in range(MAX_LENGTH):
#             decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
#             decoder_outputs.append(decoder_output)

#             if target_tensor is not None:
#                 # Teacher forcing: Feed the target as the next input
#                 decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
#             else:
#                 # Without teacher forcing: use its own predictions as the next input
#                 _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
#                 decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

#         decoder_outputs = torch.cat(decoder_outputs, dim=1)
#         decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
#         return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

#     def forward_step(self, input, hidden):
#         output = self.embedding(input)
#         output = F.relu(output)
#         output, hidden = self.rnn(output, hidden)
#         output = self.out(output)
#         return output, hidden

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

In [ ]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.rnn = nn.RNN(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2) # seq_len, batch, hidden_size -> batch, seq_len, hidden_size
        context, attn_weights = self.attention(query, encoder_outputs)
        input_rnn = torch.cat((embedded, context), dim=2)

        output, hidden = self.rnn(input_rnn, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

In [ ]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [ ]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [ ]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [ ]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [ ]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [ ]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [ ]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 21s (- 14m 5s) (5 2%) 1.8080
0m 36s (- 11m 37s) (10 5%) 0.9742
0m 52s (- 10m 46s) (15 7%) 0.5701
1m 8s (- 10m 12s) (20 10%) 0.3253
1m 24s (- 9m 49s) (25 12%) 0.1976
1m 38s (- 9m 17s) (30 15%) 0.1359
1m 53s (- 8m 53s) (35 17%) 0.1031
2m 8s (- 8m 32s) (40 20%) 0.0860
2m 22s (- 8m 11s) (45 22%) 0.0777
2m 37s (- 7m 52s) (50 25%) 0.0647
2m 52s (- 7m 34s) (55 27%) 0.0595
3m 6s (- 7m 15s) (60 30%) 0.0577
3m 20s (- 6m 57s) (65 32%) 0.0524
3m 35s (- 6m 40s) (70 35%) 0.0518
3m 49s (- 6m 22s) (75 37%) 0.0500
4m 4s (- 6m 6s) (80 40%) 0.0486
4m 18s (- 5m 49s) (85 42%) 0.0468
4m 33s (- 5m 34s) (90 45%) 0.0472
4m 47s (- 5m 17s) (95 47%) 0.0479
5m 1s (- 5m 1s) (100 50%) 0.0468
5m 16s (- 4m 46s) (105 52%) 0.0438
5m 30s (- 4m 30s) (110 55%) 0.0446
5m 43s (- 4m 14s) (115 57%) 0.0427
5m 58s (- 3m 58s) (120 60%) 0.0417
6m 12s (- 3m 43s) (125 62%) 0.0414
6m 27s (- 3m 28s) (130 65%)

In [ ]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> vous etes fort elegantes
= you re very stylish
< you re very stylish <EOS>

> vous etes trop lent
= you re too slow
< you re too slow <EOS>

> tu imagines des choses
= you re imagining things
< you re imagining things <EOS>

> nous sommes perplexes
= we re baffled
< we re confused <EOS>

> tu es tres timide
= you re very timid
< you re very timid <EOS>

> vous etes fort effrontee
= you re very forward
< you re very forward <EOS>

> il est trop occupe
= he s too busy
< he s too busy <EOS>

> il est tres honnete
= he is very honest
< he is very honest <EOS>

> nous sur reagissons
= we re overreacting
< we re trapped <EOS>

> il est tres jeune
= he is very young
< he is very young <EOS>



### Discussion

The RNN Encoder-Decoder model with Bahdanau Attention was successfully implemented for French-to-English translation. The training loss decreased steadily over 200 epochs, showing that the model learned the translation patterns effectively. Most test sentences were translated correctly, demonstrating the usefulness of the attention mechanism. However, a few translations contained minor word-choice errors, indicating that the model still has difficulty handling certain contextual meanings.


### Conclusion

This experiment successfully demonstrated the implementation and evaluation of an RNN Encoder-Decoder model with Bahdanau Attention for neural machine translation. The attention mechanism improved translation accuracy by helping the decoder focus on important input words. Although the model performed well, its accuracy can be further enhanced by using a larger dataset, optimizing hyperparameters, or employing advanced RNN architectures such as GRUs or LSTMs.
